## 1. Librerías y cargado del dataset

In [99]:
#Librerías
from google.colab import drive
import pandas as pd

In [100]:
#Carga del dataset a través de Drive
drive.mount('/content/drive')

ruta = '/content/drive/MyDrive/10ª Edición Máster en Data Science/TFM/Dataset_sin_tratar.csv'
dataset_original = pd.read_csv(ruta,encoding='Latin-1')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Exploración básica

In [101]:
print(dataset_original.shape)
dataset_original.info()

#Para observar las primera 15/300 características del dataset
list(dataset_original.columns)[:15]

(441456, 330)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 441456 entries, 0 to 441455
Columns: 330 entries, _STATE to _AIDTST3
dtypes: float64(324), int64(4), object(2)
memory usage: 1.1+ GB


['_STATE',
 'FMONTH',
 'IDATE',
 'IMONTH',
 'IDAY',
 'IYEAR',
 'DISPCODE',
 'SEQNO',
 '_PSU',
 'CTELENUM',
 'PVTRESD1',
 'COLGHOUS',
 'STATERES',
 'CELLFON3',
 'LADULT']

In [102]:
#Figuran solo 328 características porque 2 de las 330 son "objects"
dataset_original.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
_STATE,441456.0,2.996871e+01,1.603471e+01,1.0,19.0,29.0,44.0,72.0
FMONTH,441456.0,6.359676e+00,3.487131e+00,1.0,3.0,6.0,9.0,12.0
IDATE,441456.0,6.563745e+06,3.488675e+06,1012016.0,3232015.0,6242015.0,10022015.0,12312015.0
IMONTH,441456.0,6.416803e+00,3.492082e+00,1.0,3.0,6.0,10.0,12.0
IDAY,441456.0,1.449273e+01,8.335468e+00,1.0,8.0,14.0,21.0,31.0
...,...,...,...,...,...,...,...,...
_RFSEAT2,441456.0,1.824624e+00,2.360812e+00,1.0,1.0,1.0,1.0,9.0
_RFSEAT3,441456.0,1.887028e+00,2.351387e+00,1.0,1.0,1.0,1.0,9.0
_FLSHOT6,157954.0,2.290705e+00,2.518086e+00,1.0,1.0,1.0,2.0,9.0
_PNEUMO2,157954.0,2.412259e+00,2.778032e+00,1.0,1.0,1.0,2.0,9.0


In [103]:
#La variable targer será DIABETE3, siendo las clases que nos importan: 1 (SI) - 3 (NO) - 4 (PREDIABETES)

conteo_target = dataset_original['DIABETE3'].value_counts()

#El hiperparámetro Normalize devuelve la frecuencia relativa con respecto del total
porcentaje_clases_target = dataset_original['DIABETE3'].value_counts(normalize = True)*100

resultado = pd.DataFrame({
    'Conteo': conteo_target,
    'Porcentaje': porcentaje_clases_target
}).transpose()

#Las 3 clases más numerosas son las que nos interesan, aunque están muy desbalanceadas (posibilidad unificar NO y PREDIABETES, o eliminar esta última)
print(resultado)

DIABETE3              3.0           1.0          4.0          2.0         7.0  \
Conteo      372104.000000  57256.000000  7690.000000  3608.000000  598.000000   
Porcentaje      84.291504     12.970015     1.741991     0.817308    0.135463   

DIABETE3          9.0  
Conteo      193.00000  
Porcentaje    0.04372  


## 3. Limpieza y filtrado

In [104]:
#Se eliminan algunas características que no aportan valor al modelo (razonado en el documento: 1. Cribado de columnas)
columnas_para_eliminar = '''
_STATE, FMONTH, IDATE, IMONTH, IDAY, IYEAR, DISPCODE, SEQNO, _PSU, QSTVER, QSTLANG, MSCODE, _STSTR, _STRWT, _RAWRAKE, _WT2RAKE, _CLLCPWT, _DUALUSE, _DUALCOR, _LLCPWT, CTELENUM, CELLFON3,
CTELNUM1, CELLFON2, CADULT, CCLGHOUS, CSTATE, LANDLINE, NUMPHON2, CPDEMO1, INTERNET, PVTRESD1, COLGHOUS, STATERES, LADULT, NUMADULT, NUMMEN, NUMWOMEN, PVTRESD2, HHADULT, HLTHPLN1, PERSDOC2,
MEDCOST, CHECKUP1, RENTHOM1, NUMHHOL2, CHILDREN, INCOME2, CDHELP, CDSOCIAL, _CHLDCNT, _INCOMG, DIABAGE2, CVDINFR4, CVDCRHD4, CVDSTRK3, CHCKIDNY, PDIABTST, PREDIAB1, INSULIN, FEETCHK2,
DOCTDIAB, CHKHEMO3, FEETCHK, EYEEXAM, DIABEYE, DIABEDU, MARITAL, EDUCA, VETERAN3, EMPLOY1, SEATBELT, CDDISCUS, SCNTMNY1, SCNTMEL1, SCNTPAID, SCNTWRK1, SCNTLPAD, SCNTLWK1, SXORIENT, TRNSGNDR,
RCSGENDR, RCSRLTN2, EMTSUPRT, LSATISFY, ADPLEASR, ADDOWN, MISTMNT, _EDUCAG, _RFSEAT2, _RFSEAT3, _LMTWRK1, _LMTSCL1, CAREGIV1, CRGVREL1, CRGVLNG1, CRGVHRS1, CRGVPRB1, CRGVPERS, CRGVHOUS,
CRGVMST2, CRGVEXPT, CDHOUSE, CDASSIST, WEIGHT2, HEIGHT3, HTIN4, HTM4, WTKG3, _BMI5, _RFBMI5, STOPSMK2, LASTSMK2, USENOW3, _SMOKER3, _RFSMOK3, ALCDAY5, DRNK3GE5, MAXDRNKS, DRNKANY5, DROCDY3_,
_RFBING5, _DRNKWEK, _RFDRHV5, FVGREEN, ARTHEXER, FVORANG, VEGETAB1, GRENDAY_, ORNGDAY_, VEGEDA1_, _MISVEGN, _VEGLT1, _VEG23, _VEGETEX, FTJUDA1_, FRUTDA1_, FRUITJU1, FRUIT1, _MISFRTN, _FRTLT1, _FRT16,
_FRUITEX, FVBEANS, EXERANY2, EXRACT11, EXRACT21, EXEROFT2, EXERHMM2, ADMOVE, EXACTOT1, EXACTOT2, _TOTINDA, METVL11_, METVL21_, MAXVO2_, FC60_, ACTIN11_, ACTIN21_, PADUR1_, PADUR2_, PAFREQ1_,
PAFREQ2_, _MINAC11, _MINAC21, STRFREQ_, PAMIN11_, PAMIN21_, PA1MIN_, PAVIG11_, PAVIG21_, PA1VIGM_, _PA150R2, _PA300R2, _PA30021, _PASTRNG, _PAREC1, _PASTAE1, _LMTACT1, LMTJOIN3, ARTHSOCL,
JOINPAIN, PAINACT2, TOLDHI2, QLMENTL2, QLSTRES2, QLHLTH2, _RFHLTH, VIDFCLT2, VIREDIF3, VIPRFVS2, VINOCRE2, VIEYEXM2, VIINSUR2, VICTRCT4, VIGLUMA2, VIMACDG2, ASTHMAGE, ASATTACK, ASERVIST,
ASDRVIST, ASRCHKUP, ASACTLIM, ASYMPTOM, ASNOSLEP, ASTHMED3, ASINHALR, CASTHDX2, CASTHNO2, _LTASTH1, _CASTHM1, _ASTHMS1, HAREHAB1, STREHAB1, CVDASPRN, ASPUNSAF, _MICHD, RLIVPAIN, RDUCHART,
RDUCSTRK, ARTTODAY, ARTHWGT, ARTHEDU, _DRDXAR1, _RFCHOL,  _CHISPNC, _CRACE1, _CPRACE, _PRACE1, _MRACE1, _HISPANC, _RACEG21, _RACEGR3, _RACE_G1, _AGE65YR, _AGE80, _AGE_G, _RFHYPE5,
CHOLCHK, HIVTST6, HIVTSTD3, WHRTST10, HADMAM, HOWLONG, HADPAP2, LASTPAP2, HPVTEST, HPLSTTST, HADHYST2, PROFEXAM, LENGEXAM, BLDSTOOL, LSTBLDS3, HADSIGM3, HADSGCO1, LASTSIG3, PSATEST1, PSATIME,
PCPSARS1, PCPSADE1, PCDMDECN, _AIDTST3, _CHOLCHK, FLUSHOT6, FLSHTMY2, IMFVPLAC, PNEUVAC3, TETANUS, HPVADVC2, HPVADSHT, SHINGLE2, _FLSHOT6, _PNEUMO2, DRADVISE, PREGNANT, PCPSAAD2, PCPSADI1,
PCPSARE1, _HCVU651, STRENGTH, PAMISS1_, SMOKDAY2, EXERHMM1, WTCHSALT, LONGWTCH, BEANDAY_, _FRTRESP, _VEGRESP, _PAINDX1, CIMEMLOS, ASTHMA3, ASTHNOW, CHCSCNCR, CHCOCNCR, CHCCOPD1, BLDSUGAR
'''
# Se eliminan los saltos de línea
columnas_para_eliminar = columnas_para_eliminar.replace("\n", "")
# Se eliminan espacios que pueda haber en las variables
columnas_para_eliminar = columnas_para_eliminar.replace(" ", "")
# Se obtiene una lista a partir de una string
columnas_eliminar = columnas_para_eliminar.split(',')

# De las 330 columnas, se eliminarán 296 por lo que se tendría que tener un nuevo df de 34 columnas (33 variables + 1 target)
print(f'Número de columnas a eliminar: {len(columnas_eliminar)}')
# Se comprueba que no hay ninguna columna duplicada
print(f'Columnas a eliminar únicas: {len(set(columnas_eliminar))}')

#Primero se crea una compia del df original
dataset_pretratrado = dataset_original

#Se comprueba que todas las columnas existen en el df y se incluye una que no existe "prueba" para probar el código
columnas_no_existentes = [col for col in columnas_eliminar if col not in dataset_pretratrado.columns]

if columnas_no_existentes:
    print("Las siguientes columnas NO existen en el DataFrame:")
    for col in columnas_no_existentes:
        print(col)
else:
    print('Todas las columnas existen en el DataFrame')

#Se eliminan las columnas de la lista.
#Se se incluye errors='ignore' se evita que si una columna está mal escrita, haya error, aunque se comprobó en el paso previo.
dataset_pretratrado = dataset_pretratrado.drop(columns=columnas_eliminar, errors='ignore')

Número de columnas a eliminar: 296
Columnas a eliminar únicas: 296
Todas las columnas existen en el DataFrame


In [105]:
#Se comprueba que se tienen 33 variables y 1 target
dataset_pretratrado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 441456 entries, 0 to 441455
Data columns (total 34 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   GENHLTH   441454 non-null  float64
 1   PHYSHLTH  441455 non-null  float64
 2   MENTHLTH  441456 non-null  float64
 3   POORHLTH  226964 non-null  float64
 4   BPHIGH4   441455 non-null  float64
 5   BPMEDS    178188 non-null  float64
 6   BLOODCHO  441456 non-null  float64
 7   HAVARTH3  441455 non-null  float64
 8   ADDEPEV2  441456 non-null  float64
 9   DIABETE3  441449 non-null  float64
 10  SEX       441456 non-null  float64
 11  QLACTLM2  432118 non-null  float64
 12  USEEQUIP  431026 non-null  float64
 13  BLIND     430302 non-null  float64
 14  DECIDE    429716 non-null  float64
 15  DIFFWALK  429122 non-null  float64
 16  DIFFDRES  428728 non-null  float64
 17  DIFFALON  428130 non-null  float64
 18  SMOKE100  427201 non-null  float64
 19  AVEDRNK2  210838 non-null  float64
 20  EXER

In [106]:
#Se filtran las observaciones que tienen clases del target que se necesitan (3, 1, 4) con 3 clases

lista_targets = [3, 1, 4]
dataset_pretratado = dataset_pretratrado[dataset_pretratrado['DIABETE3'].isin(lista_targets)].copy()

#Se comprobó antes que tendría que haber 437.050  registros. El índice se reseteará cuando se termine de filtrar
dataset_pretratado['DIABETE3'].info()

<class 'pandas.core.series.Series'>
Index: 437050 entries, 0 to 441455
Series name: DIABETE3
Non-Null Count   Dtype  
--------------   -----  
437050 non-null  float64
dtypes: float64(1)
memory usage: 6.7 MB


In [107]:
#Se comprueban cuales son las columnas que tienen mayor número de nulos
nulls = dataset_pretratado.isnull().sum().sort_values(ascending=False)
nulls_pct = (dataset_pretratado.isnull().sum()/len(dataset_pretratado)*100).sort_values(ascending=False)
pd.DataFrame({"Nulos": nulls, "Porcentaje": nulls_pct})

,Nulos,Porcentaje
ADANXEV,416894,95.388171
ADTHINK,416856,95.379476
ADFAIL,416842,95.376273
ADEAT1,416835,95.374671
ADENERGY,416824,95.372154
ADSLEEP,416819,95.371010
ARTHDIS2,301637,69.016588
BPMEDS,260296,59.557488
AVEDRNK2,228057,52.180986
POORHLTH,212709,48.669260


In [108]:
# De las variables preseleccionadas, se eliminan las que mayor número de NaN presentan (razonado en el documento: 1.1 Segundo cribado de columnas)
columnas_eliminar_2 = ['ADSLEEP', 'ADENERGY', 'ADEAT1', 'ADFAIL', 'ADTHINK', 'ADANXEV', 'ARTHDIS2']

# Se valora conservar las columnas POORHLTH y AVEDRNK2. En este conteo, no se tienen en cuenta los NaN
porcentaje = dataset_pretratado['POORHLTH'].value_counts(normalize=True)*100
print(porcentaje.head(5))

porcentaje2 = dataset_pretratrado['AVEDRNK2'].value_counts(normalize=True)*100
print(porcentaje2.head(5))

# Lista actualizada:
columnas_eliminar_2 = ['ADSLEEP', 'ADENERGY', 'ADEAT1', 'ADFAIL', 'ADTHINK', 'ADANXEV', 'ARTHDIS2', 'POORHLTH', 'AVEDRNK2']

dataset_pretratado_2 = dataset_pretratado.drop(columns=columnas_eliminar_2, errors='ignore')

POORHLTH
88.0    55.475816
30.0     8.714412
2.0      5.415862
1.0      4.809197
5.0      3.578035
Name: proportion, dtype: float64
AVEDRNK2
1.0    46.277237
2.0    29.561559
3.0    10.741897
4.0     4.502509
5.0     2.368169
Name: proportion, dtype: float64


In [118]:
#Se comprueba que se tienen 24 variables y 1 target
dataset_pretratado_2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 437050 entries, 0 to 441455
Data columns (total 25 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   GENHLTH   437048 non-null  float64
 1   PHYSHLTH  437049 non-null  float64
 2   MENTHLTH  437050 non-null  float64
 3   BPHIGH4   437049 non-null  float64
 4   BPMEDS    176754 non-null  float64
 5   BLOODCHO  437050 non-null  float64
 6   HAVARTH3  437049 non-null  float64
 7   ADDEPEV2  437050 non-null  float64
 8   DIABETE3  437050 non-null  float64
 9   SEX       437050 non-null  float64
 10  QLACTLM2  427812 non-null  float64
 11  USEEQUIP  426734 non-null  float64
 12  BLIND     426017 non-null  float64
 13  DECIDE    425438 non-null  float64
 14  DIFFWALK  424855 non-null  float64
 15  DIFFDRES  424464 non-null  float64
 16  DIFFALON  423875 non-null  float64
 17  SMOKE100  422959 non-null  float64
 18  EXEROFT1  291201 non-null  float64
 19  _RACE     437050 non-null  float64
 20  _AGEG5YR 

In [110]:
# Se crea una clase para visualizar el % de los datos faltantes en las columnas elegidas que no son NaN
class PorcentajesValoresFaltantes:
  def __init__(self, df):
    self.df = df

  # Se calcula el porcentaje, por columna en función de los valores de la lista que representan valores faltantes
  def calcular_porcentaje(self, columnas, valores_especiales):
    resultados = {'Columna': [], 'Porcentaje': []}

    for col in columnas:
      total = self.df[col].notna().sum()
      contar = self.df[col].isin(valores_especiales).sum()
      porcentaje = (contar / total) *100
      resultados['Columna'].append(col)
      resultados['Porcentaje'].append(porcentaje)

    return pd.DataFrame(resultados)

  # Grupos_columnas (diccionario):
  # Clave es el nombre del grupo / Valor es una tupla (columnas, valores_especiales)
  def calcular_todos(self, grupo_columnas):
    resultados_globales = {}

    for nombre, (cols, vals) in grupo_columnas.items():
      resultados_globales[nombre] = self.calcular_porcentaje(cols, vals)

    return resultados_globales


In [113]:
# Se crea el objeto de la clase PorcentajesValoresFaltantes con el df
analizador = PorcentajesValoresFaltantes(dataset_pretratado_2)

# Se crea el diccionario con nombre del grupo / columnas y valores especiales
grupos_columnas = {
    '7_9': (['GENHLTH', 'BPHIGH4', 'BPMEDS', 'BLOODCHO', 'HAVARTH3', 'QLACTLM2', 'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'SMOKE100', 'ADDEPEV2', '_PACAT1'], [7, 9]),
    '77_88_99': (['PHYSHLTH', 'MENTHLTH'], [77, 88, 99]),
    '777_999': (['EXEROFT1'], [777, 999]),
    '9': (['_RACE'], [9]),
    '14': (['_AGEG5YR'], [14])
}

# Se crea la variable resultados con el metodo calcular_todos del objeto (instancia) analizador
resultados = analizador.calcular_todos(grupos_columnas)

# Se muestan los resultados individuales
for nombre, df_resultado in resultados.items():
    print(f"\n=== Resultados para grupo {nombre} ===")
    print(df_resultado)


=== Resultados para grupo 7_9 ===
     Columna  Porcentaje
0    GENHLTH    0.278459
1    BPHIGH4    0.286924
2     BPMEDS    0.176517
3   BLOODCHO    2.129047
4   HAVARTH3    0.594899
5   QLACTLM2    0.721345
6   USEEQUIP    0.249101
7      BLIND    0.380971
8     DECIDE    0.665902
9   DIFFWALK    0.517824
10  DIFFDRES    0.269281
11  DIFFALON    0.437157
12  SMOKE100    0.762722
13  ADDEPEV2    0.461732
14   _PACAT1   12.638829

=== Resultados para grupo 77_88_99 ===
    Columna  Porcentaje
0  PHYSHLTH   64.349764
1  MENTHLTH   69.970026

=== Resultados para grupo 777_999 ===
    Columna  Porcentaje
0  EXEROFT1    0.967373

=== Resultados para grupo 9 ===
  Columna  Porcentaje
0   _RACE    1.667544

=== Resultados para grupo 14 ===
    Columna  Porcentaje
0  _AGEG5YR    1.187965


In [122]:
# De las variables preseleccionadas, se eliminan las que tienen mayor número de No Sabe/No contesto, Se nego (razonado en el documento: 1.2 Tercer cribado de columnas)
columnas_eliminar_3 = ['PHYSHLTH', 'MENTHLTH']

dataset_pretratado_3 = dataset_pretratado_2.drop(columns=columnas_eliminar_3, errors='ignore')

In [123]:
#Se comprueba que se tienen 22 variables y 1 target
dataset_pretratado_3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 437050 entries, 0 to 441455
Data columns (total 23 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   GENHLTH   437048 non-null  float64
 1   BPHIGH4   437049 non-null  float64
 2   BPMEDS    176754 non-null  float64
 3   BLOODCHO  437050 non-null  float64
 4   HAVARTH3  437049 non-null  float64
 5   ADDEPEV2  437050 non-null  float64
 6   DIABETE3  437050 non-null  float64
 7   SEX       437050 non-null  float64
 8   QLACTLM2  427812 non-null  float64
 9   USEEQUIP  426734 non-null  float64
 10  BLIND     426017 non-null  float64
 11  DECIDE    425438 non-null  float64
 12  DIFFWALK  424855 non-null  float64
 13  DIFFDRES  424464 non-null  float64
 14  DIFFALON  423875 non-null  float64
 15  SMOKE100  422959 non-null  float64
 16  EXEROFT1  291201 non-null  float64
 17  _RACE     437050 non-null  float64
 18  _AGEG5YR  437050 non-null  float64
 19  _BMI5CAT  401327 non-null  float64
 20  _FRUTSUM 

In [ ]:
#IMPUTACIÓN DE NULOS Y VALORES 7 / 8 / 9 -> ONE HOT Y REINDEXAR...

## 4. Preparación para modelado